In [ ]:
!pip install datasets transformers[torch] accelerate -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 106.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("ChrisZeng/electra-large-discriminator-nli-efl-hateval")
model = AutoModelForSequenceClassification.from_pretrained("ChrisZeng/electra-large-discriminator-nli-efl-hateval")
from google.colab import userdata
userdata.get('TFG_NLP')

classifier_stereotype = pipeline(
    "text-classification",
    model="Narrativa/distilroberta-finetuned-stereotype-detection",
    truncation=True,
    max_length=512
)

ejemplos_estereotipos = [
    "Las mujeres son mejores cuidando niños que los hombres.",
    "Los ingenieros son personas muy lógicas y poco emocionales.",
    "Todos los asiáticos son buenos en matemáticas.",
    "Los hombres nunca lloran y siempre deben ser fuertes.",
    "Las personas mayores son malas con la tecnología.",
    "Este poema trata sobre el amor y la naturaleza sin mencionar género ni etnia."
]

for texto in ejemplos_estereotipos:
    resultado = classifier_stereotype(texto)[0]
    print(f"Texto: {texto}")
    print(f"→ {resultado['label']}  (score: {resultado['score']:.4f})\n")

config.json:   0%|          | 0.00/952 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Invalid model-index. Not loading eval results into CardData.
RobertaForSequenceClassification LOAD REPORT from: Narrativa/distilroberta-finetuned-stereotype-detection
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Texto: Las mujeres son mejores cuidando niños que los hombres.
→ stereotype  (score: 0.9971)

Texto: Los ingenieros son personas muy lógicas y poco emocionales.
→  anti-stereotype  (score: 0.8224)

Texto: Todos los asiáticos son buenos en matemáticas.
→ stereotype  (score: 0.9978)

Texto: Los hombres nunca lloran y siempre deben ser fuertes.
→ stereotype  (score: 0.9988)

Texto: Las personas mayores son malas con la tecnología.
→  anti-stereotype  (score: 0.7464)

Texto: Este poema trata sobre el amor y la naturaleza sin mencionar género ni etnia.
→ stereotype  (score: 0.9951)



In [ ]:
# ==============================================================================
# 1. INSTALACIÓN DE LIBRERÍAS (Ejecutar si es necesario en Colab)
# ==============================================================================
# !pip install transformers datasets accelerate

from transformers import pipeline
from google.colab import userdata


# ==============================================================================
# 2. CONFIGURACIÓN DEL MODELO NLP (NLI - EFL)
# ==============================================================================
# Opcional: Obtener tu token o variable si la necesitas para tu TFG
# _ = userdata.get('TFG_NLP')

# Creamos el pipeline de clasificación de texto
classifier_nli = pipeline(
    "text-classification",
    model="ChrisZeng/electra-large-discriminator-nli-efl-hateval",
    truncation=True,
    max_length=512
)

# Diccionario de mapeo: Traduce la lógica abstracta del NLP a etiquetas de tu investigación
mapeo_resultados = {
    "CONTRADICTION": "⚠️ ESTEREOTIPO / ODIO DETECTADO",
    "ENTAILMENT":    "✅ TEXTO LIMPIO / NEUTRO",
    "NEUTRAL":       "⚪ INDETERMINADO / NEUTRALIDAD LÓGICA"
}

# ==============================================================================
# 3. DATASET DE PRUEBA (Estereotipos y control)
# ==============================================================================
ejemplos_estereotipos = [
    "Las mujeres son mejores cuidando niños que los hombres.",
    "Los ingenieros son personas muy lógicas y poco emocionales.",
    "Todos los asiáticos son buenos en matemáticas.",
    "Los hombres nunca lloran y siempre deben ser fuertes.",
    "Las personas mayores son malas con la tecnología.",
    "Este poema trata sobre el amor y la naturaleza sin mencionar género ni etnia."
]

# Hipótesis de control que el modelo NLI evaluará contra cada texto
hipotesis_control = "This text contains hate speech, stereotypes, or bias."

# ==============================================================================
# 4. EJECUCIÓN DEL PIPELINE Y VISUALIZACIÓN
# ==============================================================================
print("="*70)
print("  RESULTADOS DEL ANÁLISIS NLP (Modelo: ELECTRA-NLI-EFL)")
print("="*70 + "\n")

for texto in ejemplos_estereotipos:
    # Formateamos la entrada combinando el Texto + Token de separación + Hipótesis
    # Esto le permite al modelo realizar la inferencia lógica correctamente
    input_nli = f"{texto} [SEP] {hipotesis_control}"

    # Predecimos
    prediccion = classifier_nli(input_nli)[0]

    label_original = prediccion['label']
    score = prediccion['score']

    # Traducimos la etiqueta lógica al dominio de tu problema
    interpretacion = mapeo_resultados.get(label_original, label_original)

    # Imprimimos los resultados en pantalla
    print(f"Texto Analizado: '{texto}'")
    print(f"→ Salida NLP Bruta: {label_original}")
    print(f"→ Conclusión TFG  : {interpretacion} (Confianza: {score:.4f})")
    print("-" * 70)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: ChrisZeng/electra-large-discriminator-nli-efl-hateval
Key                             | Status     |  | 
--------------------------------+------------+--+-
electra.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  RESULTADOS DEL ANÁLISIS NLP (Modelo: ELECTRA-NLI-EFL)

Texto Analizado: 'Las mujeres son mejores cuidando niños que los hombres.'
→ Salida NLP Bruta: contradiction
→ Conclusión TFG  : contradiction (Confianza: 0.9991)
----------------------------------------------------------------------
Texto Analizado: 'Los ingenieros son personas muy lógicas y poco emocionales.'
→ Salida NLP Bruta: contradiction
→ Conclusión TFG  : contradiction (Confianza: 0.9993)
----------------------------------------------------------------------
Texto Analizado: 'Todos los asiáticos son buenos en matemáticas.'
→ Salida NLP Bruta: contradiction
→ Conclusión TFG  : contradiction (Confianza: 0.9993)
----------------------------------------------------------------------
Texto Analizado: 'Los hombres nunca lloran y siempre deben ser fuertes.'
→ Salida NLP Bruta: contradiction
→ Conclusión TFG  : contradiction (Confianza: 0.9970)
----------------------------------------------------------------------
Texto Analizad

In [ ]:
from transformers import pipeline

# 1. Inicializamos el pipeline de ELECTRA
classifier_electra = pipeline(
    "text-classification",
    model="ChrisZeng/electra-large-discriminator-nli-efl-hateval",
    truncation=True,
    max_length=512
)

ejemplos_estereotipos = [
    "Las mujeres son mejores cuidando niños que los hombres.",
    "Los ingenieros son personas muy lógicas y poco emocionales.",
    "Todos los asiáticos son buenos en matemáticas.",
    "Los hombres nunca lloran y siempre deben ser fuertes.",
    "Las personas mayores son malas con la tecnología.",
    "Este poema trata sobre el amor y la naturaleza sin mencionar género ni etnia.",
    "Los hombres son todos unos maltratadores",
    "Debería estar prohibido ser homosexual",
    "Un mono es más listo que tu"
]

# Hipótesis de control en inglés para activar el EFL del modelo
hipotesis = "this text contains hate speech or stereotypes."

print("="*75)
print("  ANÁLISIS CUANTITATIVO DE ODIO/ESTEREOTIPO - ELECTRA NLI")
print("="*75 + "\n")

for texto in ejemplos_estereotipos:
    # Enviamos el par de textos al modelo
    prediccion = classifier_electra(texto, text_pair=hipotesis)
    resultado = prediccion[0] if isinstance(prediccion, list) else prediccion

    label = resultado['label'].upper()
    confianza = resultado['score']

    # 2. CÁLCULO MATEMÁTICO DEL PORCENTAJE DE HATE
    if label == "ENTAILMENT":
        # El modelo afirma la hipótesis de odio
        porcentaje_hate = confianza * 100
    elif label == "CONTRADICTION":
        # El modelo niega la hipótesis de odio (Hate es la probabilidad inversa)
        porcentaje_hate = (1 - confianza) * 100
    else:
        # En caso de etiquetas neutrales, la probabilidad se distribuye de otra forma
        porcentaje_hate = 50.0

    # 3. IMPRESIÓN DE RESULTADOS
    print(f"Texto: '{texto}'")
    print(f"→ Inferencia Lógica: {label} (Confianza: {confianza:.4f})")
    print(f"→ Porcentaje de HATE / SESGO: {porcentaje_hate:.2f}%")

    # Barra visual de progreso para tu informe o consola
    barra = "█" * int(porcentaje_hate / 5) + "░" * (20 - int(porcentaje_hate / 5))
    print(f"  [{barra}]")
    print("-" * 75)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: ChrisZeng/electra-large-discriminator-nli-efl-hateval
Key                             | Status     |  | 
--------------------------------+------------+--+-
electra.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ANÁLISIS CUANTITATIVO DE ODIO/ESTEREOTIPO - ELECTRA NLI

Texto: 'Las mujeres son mejores cuidando niños que los hombres.'
→ Inferencia Lógica: CONTRADICTION (Confianza: 0.9982)
→ Porcentaje de HATE / SESGO: 0.18%
  [░░░░░░░░░░░░░░░░░░░░]
---------------------------------------------------------------------------
Texto: 'Los ingenieros son personas muy lógicas y poco emocionales.'
→ Inferencia Lógica: CONTRADICTION (Confianza: 0.9985)
→ Porcentaje de HATE / SESGO: 0.15%
  [░░░░░░░░░░░░░░░░░░░░]
---------------------------------------------------------------------------
Texto: 'Todos los asiáticos son buenos en matemáticas.'
→ Inferencia Lógica: CONTRADICTION (Confianza: 0.9987)
→ Porcentaje de HATE / SESGO: 0.13%
  [░░░░░░░░░░░░░░░░░░░░]
---------------------------------------------------------------------------
Texto: 'Los hombres nunca lloran y siempre deben ser fuertes.'
→ Inferencia Lógica: CONTRADICTION (Confianza: 0.8468)
→ Porcentaje de HATE / SESGO: 15.32%
  [███░░░░░░░░░░░░░░

Ejemplos uso datasets detección